In [1]:
clean_up = True
%run stdPackages.ipynb
os.chdir(d['py'])
from multOutIO import AdjustIO, CreateIOTargets

# Estimate relative cost of waste treatment

Load full IO database, waste data, and mappings:

In [2]:
t0 = 2019
db = GpyDB(os.path.join(d['project'], 'IOdata','data', 'processedData',f'IO{t0}_s146_dur7'), ws = d['work'])
wasteData = DbFromExcel.simpleLoad(os.path.join(d['data'], 'WasteDataFull.xlsx'))
wasteMaps = DbFromExcel.simpleLoad(os.path.join(d['data'], 'wasteDataMapping.xlsx'))

Read mappings:

In [3]:
dfMaps = pd.DataFrame(wasteMaps['s'].values).ffill()
dfMaps = pd.DataFrame(dfMaps.iloc[1:].values, columns = dfMaps.iloc[0]).astype(str)

## 1. Map to common industry index

### A. Map to 121 industries

Waste data has 121 industries, IO data has 146. Both includes 5 waste industries (with syntax 3839x). We operate with a single waste industry, but we use information on the size of sub-industries "383901" to identify quantity of waste for incineration and "383903" to identify the value/cost of the waste incineration.

In [4]:
wAgg = pd.Index(['Waste'], name = 'n')
weAgg = pd.Index(['WasteE'], name = 'n')
sWaste = db('s')[db('s').str.startswith('3839')]
wInc = pd.Index(['383903'], name = 's') # incineration 
wqInc = pd.Index(['383901'], name = 's') # industry specification to identify quantities of waste transported for incineration
sMap = pd.MultiIndex.from_frame(dfMaps[['s146','s121']]).rename(['s','ss'])

*Extend industry index with values from the demand vector. Map database to 146 specification:*

In [5]:
sAll = db('vD').index.levels[0]
sNew = pd.MultiIndex.from_arrays([sAll.difference(sMap.levels[0]), sAll.difference(sMap.levels[0]).rename('ss')])
sMapFull = sMap.union(sNew)

*Apply mapping:*

In [6]:
AggDB.aggDB(db, sMapFull);

*Adjust goods index as well:*

In [7]:
nMap = sMap.union(sMap.set_levels([l+'_F' for l in sMap.levels])).rename(['n','nn'])
nNew = pd.MultiIndex.from_arrays([db('n').difference(nMap.levels[0]), db('n').difference(nMap.levels[0]).rename('nn')])
nMapFull = nMap.union(nNew)

*Apply mapping:*

In [8]:
AggDB.aggDB(db, nMapFull);

### B. Align waste data definition with model industries

*Read symbols:*

In [9]:
def readSymbol(wasteData, sheet, columns = ['i','m','s','o']):
    df = pd.DataFrame(wasteData[sheet].values)
    return pd.DataFrame(df.iloc[1:,:].values, columns = df.iloc[0]).set_index(columns)[sheet[0]]
wd = {s: readSymbol(wasteData, s) for s in ('qd','qs','vd','vs')}

*Adjust values to measure in mia. DKK (or scale of database; assumes that the materials data measures in 1000 DKK). Adjust quantities to be 100 million tons instead of kton (implies prices around 0.1-10):*

In [10]:
[wd.__setitem__(k, wd[k]*1000/db._scale) for k in ('vd','vs')];
[wd.__setitem__(k, (wd[k]/1e5) * (1e9/db._scale)) for k in ('qd','qs')];

*Update names:*

In [11]:
nameMap = {'Export': 'F', 'Import': 'F', 'Household': 'HH', 'Government': 'G'}
for k in wd:
    wd[k].index = wd[k].index.set_levels(wd[k].index.levels[0].map(lambda x: nameMap[x] if x in nameMap else x), level = 0)
    wd[k] = wd[k].rename_axis(index = {'s':'k','i':'s'}).droplevel('o')

*Check balances:*

In [12]:
df = pd.concat([wd['qd'].groupby(['m','k']).sum().rename('d'),
                wd['qs'].groupby(['m','k']).sum().rename('s')], axis = 1)
def testBalance(d, s, error = 1e-6):
    return all(abs(d.groupby(['m','k']).sum()-s.groupby(['m','k']).sum())<error)
assert all([testBalance(wd[f'{k}d'], wd[f'{k}s']) for k in ('q','v')]), "Check sum of demand equals sum of supply"

## 2. Adjust waste data

### A. Residual industry

For now, map residual industry to investment sector and treat as follows:
* Remove demand for waste materials. Increase domestic waste treatment instead. *(this is already done in the data now - so this is skipped now)*
* Map residual material supply to other industries with weights from size of capital stocks (we can only do this at the same industry level as the main database).
* Add residual demand for materials (non-waste) to industries that produce durables. 

In [13]:
wIdx = pd.Index(['W'], name = 'k')
rIdx = pd.Index(['Residual'], name = 's')

Map "residual"'s supply of materials to other industries based on size of capital stocks:

In [14]:
vK = db('vD_dur').groupby('s').sum()
vK = vK/vK.sum()
for x in ('vs','qs'):
    resSupply = stdSort(adjMultiIndex.bc(wd[x].xs(rIdx[0]), vK) * vK)
    wd[x] = adj.rc_pd(wd[x], ('not', rIdx)).add(resSupply, fill_value=0)

Map "residual"'s demand for materials to investment industries:

In [15]:
vI = adj.rc_pd(db('vD'), db('s_i')).groupby('s').sum()
vI = vI/vI.sum()
for x in ('vd','qd'):
    resDemand = stdSort(adjMultiIndex.bc(wd[x].xs('Residual'), vI) * vI)
    wd[x] = adj.rc_pd(wd[x], ('not', rIdx)).add(resDemand, fill_value=0)

*Check balances again.*

In [16]:
assert all([testBalance(wd[f'{k}d'], wd[f'{k}s']) for k in ('q','v')]), "Check sum of demand equals sum of supply"

### B. Household and government supply of (non-waste) materials

The current model structure does not include material supply (non-waste) from households and government. If these are here, we reduce exports appropriately:

In [17]:
sNoSupply = pd.Index(['HH','G'], name = 's') # what 's' does not supply anything (beyond waste)
kxW = pd.Index(['R','V'], name = 'k') # k types that are not waste
for x in ('q','v'):
    noSupply = adj.rc_pd(wd[f'{x}s'], ('and', [sNoSupply, kxW]))
    wd[f'{x}s'] = adj.rc_pd(wd[f'{x}s'], ('not', noSupply))
    wd[f'{x}d'] = wd[f'{x}d'].sub(stdSort(adjMultiIndex.bc(noSupply.groupby(['m','k']).sum(), db('s_f'))),fill_value=0)

*Check balances again.*

In [18]:
assert all([testBalance(wd[f'{k}d'], wd[f'{k}s']) for k in ('q','v')]), "Check sum of demand equals sum of supply"

### C. Import-to-exports

Set import-to-export to x% of the *minimum* of exports and imports. This ensures non-negative imports/exports.

In [19]:
share = 0.3
for x in ('q','v'):
    reduction = pd.concat([adj.rc_pd(wd[f'{x}d'], db('s_f')), adj.rc_pd(wd[f'{x}s'], db('s_f'))], axis = 1).min(axis=1) * share
    wd[f'{x}s'] = wd[f'{x}s'].sub(reduction, fill_value=0)
    wd[f'{x}d'] = wd[f'{x}d'].sub(reduction, fill_value=0)

Check feasibility of waste treatment:

In [20]:
assert (adj.rc_pd(wd['qd'].xs('W',level='k'), ('not',db('s_f'))).groupby('m').sum()-wd['qs'].xs(('F','W'), level = ('s','k'))).min()>0, "Import of waste exceeds domestic treatment"

*Adhoc adjustment: If exports exceed domestic supply of a material sort, we reduce imports and exports with the same amount to ensure this remains balanced:*

In [21]:
for k in ('v','q'):
    domesticSupply = adj.rc_pd(wd[f'{k}s'], ('not', db('s_f'))).groupby(['m','k']).sum()
    diff = domesticSupply.sub(wd[f'{k}d'].xs('F'),fill_value=0)
    wd[f'{k}d'] = (adj.rc_pd(wd[f'{k}d'], db('s_f'))+diff[diff<0]).dropna().combine_first(wd[f'{k}d'])
    wd[f'{k}s'] = (adj.rc_pd(wd[f'{k}s'], db('s_f'))+diff[diff<0]).dropna().combine_first(wd[f'{k}s'])
    wd[f'{k}d'] = wd[f'{k}d'][wd[f'{k}d']!=0]
    wd[f'{k}s'] = wd[f'{k}s'][wd[f'{k}s']!=0]

Update index levels for the following sections:

In [22]:
for k in wd:
    wd[k].index = wd[k].index.remove_unused_levels()

### D. Create aggregate version

Aggregate version uses one waste industry instead of 5:

In [23]:
sWasteMap  = pd.MultiIndex.from_product([sWaste, wAgg.rename('ss')])
sMapFull = sWasteMap.union(pd.MultiIndex.from_arrays([db('s').difference(sWasteMap.levels[0]), db('s').difference(sWasteMap.levels[0]).rename('ss')]))
wdAgg = dict.fromkeys(wd)
for k in wd:
    wdAgg[k] = adjMultiIndex.applyMult(adj.rc_pd(wd[k], sMapFull), sMapFull).groupby(['ss','m','k']).sum().rename_axis(index = {'ss':'s'}).combine_first(
                adj.rc_pd(wd[k], ('not', sMapFull)))

## 3. Adjust IO with demand/supply of non IO goods

*Specify IO inputs to use in rebalancing. We include all production industries, foreign sector, and households. The waste sector produces energy from waste incineration and other services. We start by extracting the "waste incineration" part here and add it to the definition of the supply vector instead.*

### A. Add distinction between waste/waste incineration

In [24]:
vD_wasteTot = adj.rc_pd(db('vD'), sWaste.rename('n')).groupby('s').sum()
vD_wasteE = db('vD').xs(wInc[0],level='n')
vD_waste = vD_wasteTot.sub(vD_wasteE, fill_value=0)

Add new $n$ index ('Waste', 'WasteE'):

In [25]:
vD_waste.index = pyDatabases.cartesianProductIndex([vD_waste.index, wAgg])
vD_wasteE.index = pyDatabases.cartesianProductIndex([vD_wasteE.index, weAgg])

Add supply vector:

In [26]:
vS_waste = vD_waste.groupby('n').sum()
vS_waste.index = stdSort(pyDatabases.cartesianProductIndex([vS_waste.index, wAgg.rename('s')]))
vS_wasteE = vD_wasteE.groupby('n').sum()
vS_wasteE.index = stdSort(pyDatabases.cartesianProductIndex([vS_wasteE.index, weAgg.rename('s')]))

Do the same for waste incineration from abroad (*note: No vS vector for F sector*):

In [27]:
vD_wasteTotF = adj.rc_pd(db('vD'), (sWaste+'_F').rename('n')).groupby('s').sum()
vD_wasteEF = db('vD').xs(wInc[0]+'_F', level= 'n')
vD_wasteF  = vD_wasteTotF.sub(vD_wasteEF, fill_value=0)
# add n index:
vD_wasteF.index  = pyDatabases.cartesianProductIndex([vD_wasteF.index, wAgg+'_F'])
vD_wasteEF.index = pyDatabases.cartesianProductIndex([vD_wasteEF.index, weAgg+'_F'])

Aggregate database to one waste industry:

*Sector definition:*

In [28]:
mapW_s = pd.MultiIndex.from_product([sWaste, wAgg.rename('ss')])
sMapFull = mapW_s.union(pd.MultiIndex.from_arrays([db('s').difference(sWaste), db('s').difference(sWaste).rename('ss')]))
AggDB.aggDB(db, sMapFull);

*Goods definition:*

In [29]:
mapW_n = pd.MultiIndex.from_product([sWaste.rename('n'), wAgg.rename('nn')])
mapW_n = mapW_n.union(mapW_n.set_levels([l+'_F' for l in mapW_n.levels]))
nMapFull = mapW_n.union(pd.MultiIndex.from_arrays([db('n').difference(mapW_n.levels[0]), db('n').difference(mapW_n.levels[0]).rename('nn')]))
AggDB.aggDB(db, nMapFull);

Specify a demand vector net of wasteE for all relevant industries that are part of the waste materials data:

In [30]:
vD0 = adj.rc_pd(db('vD'), ('and', [db('n_p').union(db('n_F')), wdAgg['vd'].index.levels[0]]))
vD0 = vD_waste.combine_first(vD_wasteF).combine_first(vD0)
vD0 = vD0[vD0>0]
vD0.index = vD0.index.remove_unused_levels()

### B. Add materials demand

*Define demand for non-waste materials (values):*

In [31]:
vDn_df = wdAgg['vd'].rename('vD').reset_index()
vDn_df['n'] = vDn_df['m']+'_'+vDn_df['k']
vDnRaw = vDn_df.set_index(['s','n'])['vD']

*Supply:*

In [32]:
vSn_df = wdAgg['vs'].rename('vS').reset_index()
vSn_df['n'] = vSn_df['m']+'_'+vSn_df['k']
vSnRaw = vSn_df.set_index(['s','n'])['vS']

New types of traded goods to include:

In [33]:
nmTypes = pd.Index(vDn_df['m'].unique(), name = 'n') # material types
kmTypes = pd.Index(vDn_df['k'].unique(), name = 'k') # Recycled/virgin distinction
db['m'] = nmTypes.rename('m')
db['k'] = kmTypes
db['o'] = pd.Index(['D','F'], name = 'o')

Add combinations:

In [34]:
allCombinations = pd.MultiIndex.from_product([db('m'), db('k'), db('o')]).to_frame(index = False)
allCombinations['n'] = allCombinations['m']+'_'+allCombinations['k']
allCombinations.loc[allCombinations['o'] == 'F', 'n'] = allCombinations.loc[allCombinations['o'] == 'F', 'n']+'_F'
n2m2k2o_all = stdSort(pd.MultiIndex.from_frame(allCombinations))

### C. Adjust IO data 

In [35]:
db['dom2for'] = pd.MultiIndex.from_arrays([db('n_p').sort_values(), db('n_F').sort_values().rename('nn')])

Solve for adjusted IO system:

In [36]:
pd.set_option('future.no_silent_downcasting', True)
targets = CreateIOTargets(db)(vD0, vDnRaw, vSnRaw)
adjustedData = AdjustIO(ws = db.ws)(targets.db('vD0'), targets.db('vD'), targets.db('vDn'), 
                                    targets.db('vS0'), targets.db('vS'), targets.db('vSn'))

The ```AdjustIO``` class creates adjustments to IO data. For the model implementation later, we add a number of subsets to set up the nesting structure: 
* ```nm```: Full set of traded goods that requires demand/supply equations and equilibrium constraints.
* ```nm_F```: Foreign subset of ```nm```.
* ```nm_D```: Domestic subset of ```nm```.
* ```ns_F```: Foreign IO goods. *This includes waste incineration*.
* ```n_F```: All foreign goods (materials and IO goods).
* ```n2m2k2o```: Mapping.

In [34]:
db['nm'] = adjustedData('vSn').index.levels[-1] # materials
db['nm_F'] = adjustedData('vSn').xs('F').index # foreign materials
db['nm_D'] = db('nm').difference(db('nm_F')) # domestic materials
db['ns_F'] = db('n_F').copy().union(vD_wasteEF.index.levels[-1]) # foreign IO materials
db['n_F'] = db('ns_F').union(db('nm_F')) # foreign goods
db['n2m2k2o'] = adj.rc_pd(n2m2k2o_all, db('nm'))

*Values:* 
* The demand vector is extended with demand for materials, energy from waste incineration, and existing "IO" demand is replaced with adjusted values. *Note: This vector does not include value of durables.*
* The supply vector is: (1) extended with materials and waste incineration, (2) existing IO supply is *adjusted* (we do not use ```vSsolve``` directly here, only changes, because it does not cover everything), (3) foreign supply is not used (it only enters the model through domestic demand for foreign goods). 

In [35]:
vDnew = pd.concat([stdSort(adjMultiIndex.bc(pd.concat([adjustedData('vDn'), adjustedData('vDsolve')], axis = 0), db('t0'))),
                   vD_wasteE, vD_wasteEF], axis = 0)
db['vD'] = vDnew.combine_first(db('vD'))
# supply:
vSadjustments = stdSort(adjMultiIndex.bc(adj.rc_pd(adjustedData('vS0')-adjustedData('vSsolve'), ('not', db('s_f'))), db('t0')))
vSadjustments = vSadjustments.add(vS_wasteE.droplevel('n'),fill_value=0)
vSio = adj.rc_pd(db('vS'), vSadjustments).sub(vSadjustments, fill_value=0)
vSn  = stdSort(adjMultiIndex.bc(adj.rc_pd(adjustedData('vSn'), ('not', db('s_f'))), db('t0')))
vS = pd.concat([vSio, vSn, vS_wasteE], axis = 0)
db['vS'] = vS.combine_first(db('vS'))

*Prices:* For prices, we use default prices (p = 1) for all IO goods including waste incineration. For materials, we use implied prices from materials data:

In [36]:
pm = (wdAgg['vs'] / wdAgg['qs']).dropna().rename('p').reset_index()
pm['n'] = pm['m']+'_'+pm['k']
pm.loc[pm['s'] == 'F','n'] = pm.loc[pm['s'] == 'F','n']+'_F'
pm = adj.rc_pd(pm.set_index('n')['p'], db('nm')).groupby('n').mean()
pm = stdSort(adjMultiIndex.bc(pm, db('t0')))
pns = stdSort(adjMultiIndex.bc(pd.Series(1, index = db('n_p').union(vD_wasteE.index.levels[-1]).union(db('ns_F'))), db('t0')))
pn = pd.concat([pm, pns], axis = 0)
pn = pd.concat([pn, adj.rc_pd(db('p'), ('not', pn))], axis = 0)
db['p'] = pn.combine_first(db('p')) # update prices

*Quantities:* Non-waste quantities are implied from values / prices. Waste quantities treated further below.

In [37]:
db['qD'] = (db('vD')/pn).dropna().combine_first(db('qD'))
db['qS'] = (db('vS')/pn).dropna().combine_first(db('qS'))

*Trade mappings:*

In [38]:
db['dom2for'] = db('dom2for').union(pd.MultiIndex.from_arrays([db('nm_D'), (db('nm_D')+'_F').rename('nn')]))
db['dExport'] = adj.rc_pd(db('vD'), db('s_f')).index.droplevel('t').unique()
[db.series.__delitem__(k) for k in ('dImport', 'dImport_dom', 'dImport_for') if k in db.symbols];

## 3. Waste generation and treatment

If we have external data on effective recycling rates (across all industries, that is), use it here to scale up/down estimates: